In [3]:
import pandas as pd
import numpy as np

from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests

# Raw Data 불러오기
df = pd.read_csv("../01_raw/uci-secom.csv")

#공정 측정 Feature 0 ~ 589
sensor_cols = [str(i) for i in range(590)]

print(df.shape)

(1567, 592)


In [ ]:
%pip install scipy statsmodels

In [4]:
anova_results = []

for col in sensor_cols:

    # Pass / Fail 그룹 분리 + NaN 제외
    pass_values = df.loc[df["Pass/Fail"] == -1, col].dropna()
    fail_values = df.loc[df["Pass/Fail"] == 1, col].dropna()

    # NaN을 제외한 실제 관측값이 전체적으로 한 종류뿐이라면
    # ANOVA로 값 차이를 검정할 의미가 없음
    if df[col].dropna().nunique() <= 1:
        f_stat = 0.0
        p_value = 1.0
        eta_squared = 0.0

    else:
        # One-way ANOVA
        f_stat, p_value = f_oneway(pass_values, fail_values)

        # Effect Size: Eta Squared 계산
        n_pass = len(pass_values)
        n_fail = len(fail_values)

        mean_pass = pass_values.mean()
        mean_fail = fail_values.mean()

        grand_mean = (
            pass_values.sum() + fail_values.sum()
        ) / (n_pass + n_fail)

        ss_between = (
            n_pass * (mean_pass - grand_mean) ** 2
            + n_fail * (mean_fail - grand_mean) ** 2
        )

        ss_total = (
            ((pass_values - grand_mean) ** 2).sum()
            + ((fail_values - grand_mean) ** 2).sum()
        )

        eta_squared = (
            ss_between / ss_total
            if ss_total > 0
            else 0.0
        )

    anova_results.append({
        "Feature_ID": col,
        "Pass_N": len(pass_values),
        "Fail_N": len(fail_values),
        "Pass_Mean": pass_values.mean(),
        "Fail_Mean": fail_values.mean(),
        "F_stat": f_stat,
        "p_value": p_value,
        "Eta_Squared": eta_squared
    })

anova_df = pd.DataFrame(anova_results)

display(
    anova_df
    .sort_values("p_value")
    .head(20)
)

,Feature_ID,Pass_N,Fail_N,Pass_Mean,Fail_Mean,F_stat,p_value,Eta_Squared
59,59,1456,104,2.563464,8.515123,38.757060,6.158394e-10,0.024272
103,103,1461,104,-0.009913,-0.008053,36.569986,1.839039e-09,0.022862
510,510,1461,104,54.440591,74.347949,27.543103,1.747062e-07,0.017317
348,348,1439,104,0.024291,0.030448,26.565104,2.877548e-07,0.016947
431,431,1462,103,21.191221,38.924103,23.165815,1.629758e-06,0.014605
434,434,1462,103,13.719237,29.136890,19.897039,8.756522e-06,0.012570
430,430,1462,103,17.368450,33.370198,19.167584,1.276772e-05,0.012115
435,435,1462,103,8.376004,23.489069,18.816594,1.531226e-05,0.011896
21,21,1462,103,-5636.437585,-5362.274272,18.615053,1.699791e-05,0.011770
28,28,1462,103,69.598032,68.101406,18.188438,2.120768e-05,0.011503


In [7]:
# Benjamini-Hochberg FDR 보정
bh_reject, bh_pvalues, _, _ = multipletests(
    anova_df["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

anova_df["BH_Adjusted_p"] = bh_pvalues
anova_df["BH_Significant"] = bh_reject


# Bonferroni 보정
bonf_reject, bonf_pvalues, _, _ = multipletests(
    anova_df["p_value"],
    alpha=0.05,
    method="bonferroni"
)

anova_df["Bonferroni_Adjusted_p"] = bonf_pvalues
anova_df["Bonferroni_Significant"] = bonf_reject

In [8]:
anova_df = anova_df.sort_values(
    "p_value",
    ascending=True
)

display(anova_df.head(20))

print(
    "BH Significant Features:",
    anova_df["BH_Significant"].sum()
)

print(
    "Bonferroni Significant Features:",
    anova_df["Bonferroni_Significant"].sum()
)

,Feature_ID,Pass_N,Fail_N,Pass_Mean,Fail_Mean,F_stat,p_value,Eta_Squared,BH_Adjusted_p,BH_Significant,Bonferroni_Adjusted_p,Bonferroni_Significant
59,59,1456,104,2.563464,8.515123,38.757060,6.158394e-10,0.024272,3.633453e-07,True,3.633453e-07,True
103,103,1461,104,-0.009913,-0.008053,36.569986,1.839039e-09,0.022862,5.425165e-07,True,1.085033e-06,True
510,510,1461,104,54.440591,74.347949,27.543103,1.747062e-07,0.017317,3.435889e-05,True,1.030767e-04,True
348,348,1439,104,0.024291,0.030448,26.565104,2.877548e-07,0.016947,4.244384e-05,True,1.697754e-04,True
431,431,1462,103,21.191221,38.924103,23.165815,1.629758e-06,0.014605,1.923114e-04,True,9.615571e-04,True
434,434,1462,103,13.719237,29.136890,19.897039,8.756522e-06,0.012570,8.610580e-04,True,5.166348e-03,True
430,430,1462,103,17.368450,33.370198,19.167584,1.276772e-05,0.012115,1.076137e-03,True,7.532957e-03,True
435,435,1462,103,8.376004,23.489069,18.816594,1.531226e-05,0.011896,1.114307e-03,True,9.034234e-03,True
21,21,1462,103,-5636.437585,-5362.274272,18.615053,1.699791e-05,0.011770,1.114307e-03,True,1.002877e-02,True
28,28,1462,103,69.598032,68.101406,18.188438,2.120768e-05,0.011503,1.208917e-03,True,1.251253e-02,True


BH Significant Features: 36
Bonferroni Significant Features: 15


In [9]:
# BH-FDR을 통과한 Feature만 선택
bh_candidates = anova_df[
    anova_df["BH_Significant"]
].copy()

# Effect Size가 큰 순서대로 정렬
bh_candidates = bh_candidates.sort_values(
    "Eta_Squared",
    ascending=False
)

# 주요 결과 확인
display(
    bh_candidates[
        [
            "Feature_ID",
            "F_stat",
            "p_value",
            "BH_Adjusted_p",
            "Bonferroni_Adjusted_p",
            "Eta_Squared",
            "Pass_Mean",
            "Fail_Mean",
            "Pass_N",
            "Fail_N"
        ]
    ].head(20)
)

,Feature_ID,F_stat,p_value,BH_Adjusted_p,Bonferroni_Adjusted_p,Eta_Squared,Pass_Mean,Fail_Mean,Pass_N,Fail_N
59,59,38.757060,6.158394e-10,3.633453e-07,3.633453e-07,0.024272,2.563464,8.515123,1456,104
103,103,36.569986,1.839039e-09,5.425165e-07,1.085033e-06,0.022862,-0.009913,-0.008053,1461,104
510,510,27.543103,1.747062e-07,3.435889e-05,1.030767e-04,0.017317,54.440591,74.347949,1461,104
348,348,26.565104,2.877548e-07,4.244384e-05,1.697754e-04,0.016947,0.024291,0.030448,1439,104
431,431,23.165815,1.629758e-06,1.923114e-04,9.615571e-04,0.014605,21.191221,38.924103,1462,103
434,434,19.897039,8.756522e-06,8.610580e-04,5.166348e-03,0.012570,13.719237,29.136890,1462,103
430,430,19.167584,1.276772e-05,1.076137e-03,7.532957e-03,0.012115,17.368450,33.370198,1462,103
435,435,18.816594,1.531226e-05,1.114307e-03,9.034234e-03,0.011896,8.376004,23.489069,1462,103
21,21,18.615053,1.699791e-05,1.114307e-03,1.002877e-02,0.011770,-5636.437585,-5362.274272,1462,103
28,28,18.188438,2.120768e-05,1.208917e-03,1.251253e-02,0.011503,69.598032,68.101406,1462,103


In [10]:
print("BH Candidates:", len(bh_candidates))

print(
    "Bonferroni among BH:",
    bh_candidates["Bonferroni_Significant"].sum()
)

BH Candidates: 36
Bonferroni among BH: 15


In [11]:
bh_candidates["Effect_Size_Rank"] = (
    bh_candidates["Eta_Squared"]
    .rank(method="min", ascending=False)
)

feature_59 = bh_candidates[
    bh_candidates["Feature_ID"] == "59"
]

display(
    feature_59[
        [
            "Feature_ID",
            "p_value",
            "BH_Adjusted_p",
            "Eta_Squared",
            "Effect_Size_Rank"
        ]
    ]
)

,Feature_ID,p_value,BH_Adjusted_p,Eta_Squared,Effect_Size_Rank
59,59,6.158394e-10,3.633453e-07,0.024272,1.0


In [12]:
bh_candidates.to_csv(
    "../03_output/anova_bh_candidates.csv",
    index=False
)

In [13]:
anova_df.to_csv(
    "../03_output/anova_screening_summary.csv",
    index=False
)

In [14]:
# BH-FDR 통과 후보만 추출
bh_candidates = anova_df[
    anova_df["BH_Significant"]
].copy()

# Effect Size 순위
bh_candidates["Effect_Size_Rank"] = (
    bh_candidates["Eta_Squared"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# 보수적 기준까지 통과했는지 구분
bh_candidates["Screening_Tier"] = np.where(
    bh_candidates["Bonferroni_Significant"],
    "Tier A - Bonferroni",
    "Tier B - BH-FDR only"
)

# Effect Size 큰 순서로 정렬
bh_candidates = bh_candidates.sort_values(
    ["Eta_Squared", "BH_Adjusted_p"],
    ascending=[False, True]
)

display(
    bh_candidates[
        [
            "Feature_ID",
            "Effect_Size_Rank",
            "Eta_Squared",
            "F_stat",
            "p_value",
            "BH_Adjusted_p",
            "Bonferroni_Adjusted_p",
            "Screening_Tier",
            "Pass_Mean",
            "Fail_Mean",
            "Pass_N",
            "Fail_N"
        ]
    ]
)

,Feature_ID,Effect_Size_Rank,Eta_Squared,F_stat,p_value,BH_Adjusted_p,Bonferroni_Adjusted_p,Screening_Tier,Pass_Mean,Fail_Mean,Pass_N,Fail_N
59,59,1,0.024272,38.757060,6.158394e-10,3.633453e-07,3.633453e-07,Tier A - Bonferroni,2.563464,8.515123,1456,104
103,103,2,0.022862,36.569986,1.839039e-09,5.425165e-07,1.085033e-06,Tier A - Bonferroni,-0.009913,-0.008053,1461,104
510,510,3,0.017317,27.543103,1.747062e-07,3.435889e-05,1.030767e-04,Tier A - Bonferroni,54.440591,74.347949,1461,104
348,348,4,0.016947,26.565104,2.877548e-07,4.244384e-05,1.697754e-04,Tier A - Bonferroni,0.024291,0.030448,1439,104
431,431,5,0.014605,23.165815,1.629758e-06,1.923114e-04,9.615571e-04,Tier A - Bonferroni,21.191221,38.924103,1462,103
434,434,6,0.012570,19.897039,8.756522e-06,8.610580e-04,5.166348e-03,Tier A - Bonferroni,13.719237,29.136890,1462,103
430,430,7,0.012115,19.167584,1.276772e-05,1.076137e-03,7.532957e-03,Tier A - Bonferroni,17.368450,33.370198,1462,103
435,435,8,0.011896,18.816594,1.531226e-05,1.114307e-03,9.034234e-03,Tier A - Bonferroni,8.376004,23.489069,1462,103
21,21,9,0.011770,18.615053,1.699791e-05,1.114307e-03,1.002877e-02,Tier A - Bonferroni,-5636.437585,-5362.274272,1462,103
28,28,10,0.011503,18.188438,2.120768e-05,1.208917e-03,1.251253e-02,Tier A - Bonferroni,69.598032,68.101406,1462,103


In [15]:
bh_candidates.to_csv(
    "../03_output/anova_bh_candidates.csv",
    index=False
)

## Phase 2 — Spotfire EDA & Statistical Screening Summary

- Raw Data 기준 Pass 1,463개(93.36%), Fail 104개(6.64%)로 뚜렷한 class imbalance를 확인하였다.
- 전체 Missing Rate는 4.52%였으나, 일부 Feature에서는 90% 이상의 높은 결측률이 관찰되어 결측이 변수별로 불균등하게 분포함을 확인하였다.
- Time Trend EDA에서는 대표 Feature 0의 뚜렷한 level shift 또는 dispersion change를 육안상 확인하지 못했다. (Feature 0을 선택한 이유는 cherry picking을 막고 공정하게 무작위 열을 고르기 위해서이다. 제일 첫번째 열이므로 공정함.)
- 월별 Fail Rate를 추가 확인하여 Fail 건수와 전체 생산 샘플 수의 영향을 구분하였다.
- 590개 공정 측정 변수에 대해 Pass/Fail 그룹 간 one-way ANOVA screening을 수행하였다.
- Python으로 ANOVA를 재현하여 Spotfire 결과와 교차검증하였다.
- 590개의 동시 검정에 따른 multiple testing 문제를 고려하여 Benjamini-Hochberg FDR 및 Bonferroni correction을 적용하였다.
- BH-FDR 기준 36개, Bonferroni 기준 15개의 통계적 screening 후보가 확인되었다.
- 가장 높은 효과크기를 보인 Feature 59의 Eta Squared는 약 0.024로, 매우 작은 p-value와 달리 실제 그룹 차이의 크기는 제한적이었다.
- 따라서 statistical significance와 practical effect size를 구분하여 해석하였으며, ANOVA 후보를 불량 원인이나 최종 중요 변수로 단정하지 않았다.
- 본 결과는 이후 다변량 ML 모델의 Feature Importance 결과와 비교하기 위한 독립적인 EDA screening 결과로 사용한다.